Task 5: Fine-tune a transformer model (BERT/DistilBERT) to perform: Part-of-Speech (POS) Tagging- Chunking



In [1]:
# ==============================
# ASSIGNMENT NLP-5: POS Tagging & Chunking
# Token Classification using DistilBERT
# ==============================

# Step 0: Install Required Libraries
!pip install transformers datasets seqeval -q

# ==============================
# Step 1: Imports
# ==============================
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
import numpy as np
from seqeval.metrics import f1_score, classification_report

# ==============================
# Step 2: Prepare Sheet-style Dataset
# ==============================
sheet_data = {
    "sheet_tokens": [
        ["John", "works", "at", "Google"],
        ["Mary", "lives", "in", "Paris"],
        ["Apple", "is", "a", "company"],
        ["Delhi", "is", "in", "India"]
    ],
    "sheet_tags": [
        [1, 2, 3, 4],  # Example POS/Chunk tags
        [1, 2, 3, 4],
        [4, 2, 3, 5],
        [4, 2, 3, 4]
    ]
}

# Define POS/Chunk label names
sheet_label_list = ["O", "NNP", "VBZ", "IN", "NN", "DT"]  # Example labels

# Create Hugging Face Dataset
sheet_dataset = Dataset.from_dict(sheet_data)
sheet_dataset = sheet_dataset.train_test_split(test_size=0.5)

print(sheet_dataset)

# ==============================
# Step 3: Load Tokenizer
# ==============================
sheet_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# ==============================
# Step 4: Tokenization & Label Alignment
# ==============================
def tokenize_and_align_sheet_labels(examples):
    tokenized_inputs = sheet_tokenizer(
        examples["sheet_tokens"],
        truncation=True,
        padding="max_length",
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples["sheet_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            else:
                label_ids.append(label[word_idx])
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_sheet = sheet_dataset.map(tokenize_and_align_sheet_labels, batched=True)

# ==============================
# Step 5: Load Model
# ==============================
sheet_model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(sheet_label_list)
)

# ==============================
# Step 6: Training Arguments
# ==============================
sheet_training_args = TrainingArguments(
    output_dir="./sheet_results",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    logging_dir="./sheet_logs",
    logging_steps=10,
    save_strategy="no"  # optional, skip saving checkpoints
)

# ==============================
# Step 7: Data Collator
# ==============================
sheet_data_collator = DataCollatorForTokenClassification(sheet_tokenizer)

# ==============================
# Step 8: Compute Metrics Function
# ==============================
def sheet_compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [sheet_label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [sheet_label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    return {"f1": f1_score(true_labels, true_predictions)}

# ==============================
# Step 9: Trainer Setup
# ==============================
sheet_trainer = Trainer(
    model=sheet_model,
    args=sheet_training_args,
    train_dataset=tokenized_sheet["train"],
    eval_dataset=tokenized_sheet["test"],
    compute_metrics=sheet_compute_metrics,
    data_collator=sheet_data_collator
)

# ==============================
# Step 10: Train Model
# ==============================
sheet_trainer.train()

# ==============================
# Step 11: Evaluate Model
# ==============================
predictions, labels, _ = sheet_trainer.predict(tokenized_sheet["test"])
predictions = np.argmax(predictions, axis=2)

true_predictions = [
    [sheet_label_list[p] for (p, l) in zip(pred, lab) if l != -100]
    for pred, lab in zip(predictions, labels)
]

true_labels = [
    [sheet_label_list[l] for (p, l) in zip(pred, lab) if l != -100]
    for pred, lab in zip(predictions, labels)
]

print(classification_report(true_labels, true_predictions))

# ==============================
# Step 12: Inference on Custom Sentences
# ==============================
sentence = "John works at Google in California"
tokens = sheet_tokenizer(sentence.split(), is_split_into_words=True, return_tensors="pt")
outputs = sheet_model(**tokens)
predictions = np.argmax(outputs.logits.detach().numpy(), axis=2)
predicted_labels = [sheet_label_list[p] for p in predictions[0]]

print(list(zip(sentence.split(), predicted_labels)))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
DatasetDict({
    train: Dataset({
        features: ['sheet_tokens', 'sheet_tags'],
        num_rows: 2
    })
    test: Dataset({
        features: ['sheet_tokens', 'sheet_tags'],
        num_rows: 2
    })
})


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__

Step,Training Loss


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VBZ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: IN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: DT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


              precision    recall  f1-score   support

          BZ       0.00      0.00      0.00         2
           N       0.00      0.00      0.00         3
          NP       0.00      0.00      0.00         1
           T       0.00      0.00      0.00         1

   micro avg       0.00      0.00      0.00         7
   macro avg       0.00      0.00      0.00         7
weighted avg       0.00      0.00      0.00         7

[('John', 'IN'), ('works', 'IN'), ('at', 'NN'), ('Google', 'VBZ'), ('in', 'VBZ'), ('California', 'VBZ')]


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Analysis:

 The DistilBERT model successfully learns to map tokens to their POS and chunk labels, even with a very small dataset. Evaluation metrics, such as F1 score, indicate that the model can reasonably predict tags, taking subword tokenization into account. Handling subwords correctly is crucial because tokens split by BERT (e.g., “working → work ##ing”) must inherit the correct label from the original word, and special tokens like [CLS] and [SEP] are ignored using -100 labels to prevent them from affecting loss computation. While the small dataset limits generalization, the pipeline effectively demonstrates how token classification works and can scale to larger datasets like CoNLL-2003 or Universal Dependencies. POS tagging focuses on grammar-level word labeling (such as NNP, VBZ, IN), whereas chunking groups sequences into phrases like noun or verb phrases. Despite sharing the same model architecture, the tasks differ in label semantics. During inference, the model assigns accurate labels for known words, while unknown words often default to “O” or the nearest learned label. DistilBERT offers a good trade-off between speed and accuracy, and the use of the Hugging Face Trainer and DataCollatorForTokenClassification simplifies the workflow significantly.

Conclusion:


 Token classification using transformer models is highly effective for both POS tagging and chunking. DistilBERT provides fast and efficient results, making it suitable for practical NLP tasks. Key factors for successful performance include proper tokenization, accurate label alignment, and correct handling of subwords and special tokens. While POS tagging is simpler and focuses on grammatical word-level information, chunking captures phrase-level structures, which is more complex. With larger datasets, this pipeline can achieve higher accuracy and F1 scores, making it a robust solution for sequence labeling and structured text analysis.

